In [ ]:
# BigAlpha 2026 提交 - 高频因子加权组合
# 用户指定的源因子 notebook:
# - submission/高频因子submission2/高频2_rank4.ipynb
# - submission/高频因子submission2/高频2_rank12.ipynb
# - submission/高频因子submission2/高频2_rank69.ipynb
# - submission/高频因子submission2/高频2_rank77.ipynb
# - submission/高频因子submission3/高频3_rank35.ipynb
# 方法: 子因子最终方向 -> 每日横截面 rank(0-1) -> 按指定权重平均(skipna).
# OOM 控制: 92 个日历日一块；块内逐因子查询；只保留股票池内的 rank 长表；块结束立即聚合和释放。
import gc
import time

import numpy as np
import pandas as pd


FACTOR_SPECS = {
    'hf2_rank4': {'sql': r'''
WITH raw_factor AS (

WITH

src AS (
    SELECT
        date::DATE::DATETIME AS trading_day,
        instrument::string AS instrument,
        date AS bar_time,
        open,
        high,
        low,
        close,
        pre_close,
        amount,
        volume,
        deal_number,
        ROW_NUMBER() OVER (PARTITION BY date::DATE::DATETIME, instrument ORDER BY date) AS minute_no
    FROM __BAR1M__
    WHERE date BETWEEN '__START__' AND '__END__'
),
diffed AS (
    SELECT
        *,
        GREATEST(amount - COALESCE(LAG(amount) OVER w, 0), 0) AS amount_1m,
        GREATEST(volume - COALESCE(LAG(volume) OVER w, 0), 0) AS volume_1m,
        GREATEST(deal_number - COALESCE(LAG(deal_number) OVER w, 0), 0) AS num_trades_1m,
        LAG(close) OVER w AS prev_min_close
    FROM src
    WINDOW w AS (PARTITION BY trading_day, instrument ORDER BY bar_time)
),
signed AS (
    SELECT
        *,
        CASE
            WHEN close > prev_min_close THEN volume_1m
            WHEN close < prev_min_close THEN -volume_1m
            ELSE 0
        END AS obv_step,
        (close / NULLIF(pre_close, 0) - 1) * volume_1m AS pvt_step
    FROM diffed
),
features AS (
    SELECT
        trading_day AS date,
        instrument,
        bar_time,
        minute_no,
        COUNT(*) OVER (PARTITION BY trading_day, instrument) AS minute_count,
        open,
        high,
        low,
        close,
        CAST(volume_1m AS DOUBLE) AS volume,
        CAST(amount_1m AS DOUBLE) AS amount,
        CAST(close / NULLIF(pre_close, 0) - 1 AS DOUBLE) AS return,
        CAST(SUM(amount_1m) OVER day_to_now / NULLIF(SUM(volume_1m) OVER day_to_now, 0) AS DOUBLE) AS vwap,
        CAST(AVG(close) OVER day_to_now AS DOUBLE) AS twap,
        CAST(SUM(obv_step) OVER day_to_now AS DOUBLE) AS obv,
        CAST(SUM(pvt_step) OVER day_to_now AS DOUBLE) AS pvt,
        CAST(num_trades_1m AS DOUBLE) AS num_trades,
        CAST(volume_1m / NULLIF(num_trades_1m, 0) AS DOUBLE) AS vol_per_trade,
        CAST(amount_1m / NULLIF(num_trades_1m, 0) AS DOUBLE) AS amt_per_trade,
        CAST(AVG(close) OVER last5 AS DOUBLE) AS close_ma5,
        CAST(AVG(close) OVER last15 AS DOUBLE) AS close_ma15,
        CAST(AVG(close) OVER last30 AS DOUBLE) AS close_ma30,
        CAST(AVG(close / NULLIF(pre_close, 0) - 1) OVER last5 AS DOUBLE) AS return_ma5,
        CAST(AVG(close / NULLIF(pre_close, 0) - 1) OVER last15 AS DOUBLE) AS return_ma15,
        CAST(AVG(close / NULLIF(pre_close, 0) - 1) OVER last30 AS DOUBLE) AS return_ma30,
        CAST(AVG(volume_1m) OVER last5 AS DOUBLE) AS volume_ma5,
        CAST(AVG(volume_1m) OVER last15 AS DOUBLE) AS volume_ma15,
        CAST(AVG(volume_1m) OVER last30 AS DOUBLE) AS volume_ma30,
        CAST(AVG(volume_1m / NULLIF(num_trades_1m, 0)) OVER last5 AS DOUBLE) AS vol_per_trade_ma5,
        CAST(AVG(volume_1m / NULLIF(num_trades_1m, 0)) OVER last15 AS DOUBLE) AS vol_per_trade_ma15,
        CAST(AVG(volume_1m / NULLIF(num_trades_1m, 0)) OVER last30 AS DOUBLE) AS vol_per_trade_ma30
    FROM signed
    WINDOW
        day_to_now AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
        last5 AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN 4 PRECEDING AND CURRENT ROW),
        last15 AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN 14 PRECEDING AND CURRENT ROW),
        last30 AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN 29 PRECEDING AND CURRENT ROW)
)
,
picked AS (
    SELECT
        *,
        volume_ma15 AS A_value,
        num_trades AS B_raw,
        twap AS mask_value,
        CASE
            WHEN NULL IS NULL THEN 1
            WHEN 0.5 IS NULL THEN GREATEST(minute_count - NULL + 1, 1)
            ELSE GREATEST(CAST(ROUND(1 + 0.5 * (minute_count - 1) - NULL / 2.0) AS INTEGER), 1)
        END AS start_minute,
        CASE
            WHEN NULL IS NULL THEN minute_count
            WHEN 0.5 IS NULL THEN minute_count
            ELSE LEAST(CAST(ROUND(1 + 0.5 * (minute_count - 1) + NULL / 2.0) AS INTEGER), minute_count)
        END AS end_minute
    FROM features
),
windowed AS (
    SELECT
        *,

        B_raw AS B_value
    FROM picked
    WHERE minute_no BETWEEN start_minute AND end_minute
),
ranked AS (
    SELECT
        *,
        (RANK() OVER (PARTITION BY date, instrument ORDER BY mask_value) - 1.0)
            / NULLIF(COUNT(*) OVER (PARTITION BY date, instrument) - 1, 0) AS mask_rank,
        (RANK() OVER (PARTITION BY date, instrument ORDER BY A_value) - 1.0)
            / NULLIF(COUNT(*) OVER (PARTITION BY date, instrument) - 1, 0) AS A_rank,
        MAX(minute_no) OVER (PARTITION BY date, instrument) AS max_minute_no,
        (minute_no - MIN(minute_no) OVER (PARTITION BY date, instrument))
            / NULLIF(MAX(minute_no) OVER (PARTITION BY date, instrument) - MIN(minute_no) OVER (PARTITION BY date, instrument), 0) AS pos01,
        AVG(A_value) OVER (PARTITION BY date, instrument) AS A_mean,
        nanstd(A_value) OVER (PARTITION BY date, instrument) AS A_std,
        AVG(B_value) OVER (PARTITION BY date, instrument) AS B_mean,
        nanstd(B_value) OVER (PARTITION BY date, instrument) AS B_std
    FROM windowed
),
prepared AS (
    SELECT
        *,
        mask_rank <= 0.5 AS keep_mask,
        (A_value - A_mean) / NULLIF(A_std, 0) AS A_z,
        (B_value - B_mean) / NULLIF(B_std, 0) AS B_z,
        MAX(A_value) OVER (PARTITION BY date, instrument) AS A_max,
        MIN(A_value) OVER (PARTITION BY date, instrument) AS A_min
    FROM ranked
),
factor_raw AS (
    SELECT
        1 AS factor_id,
        date,
        instrument,
        regr_slope(CASE WHEN keep_mask THEN A_value END, CASE WHEN keep_mask THEN B_value END) AS factor_raw,
        COUNT(CASE WHEN keep_mask THEN 1 END) AS mask_n
    FROM prepared
    GROUP BY date, instrument
)
SELECT factor_id, date, instrument, factor_raw, mask_n
FROM factor_raw
WHERE factor_raw IS NOT NULL

)
SELECT
    date,
    instrument,
    factor_raw * -1 AS factor
FROM raw_factor
WHERE factor_raw IS NOT NULL
''', 'formula': '2|volume_ma15|num_trades|All|0.5|twap|low_0.5||Slope|0', 'direction': '-1', 'source': 'submission/高频因子submission2/高频2_rank4.ipynb'},
    'hf2_rank12': {'sql': r'''
WITH raw_factor AS (

WITH

src AS (
    SELECT
        date::DATE::DATETIME AS trading_day,
        instrument::string AS instrument,
        date AS bar_time,
        open,
        high,
        low,
        close,
        pre_close,
        amount,
        volume,
        deal_number,
        ROW_NUMBER() OVER (PARTITION BY date::DATE::DATETIME, instrument ORDER BY date) AS minute_no
    FROM __BAR1M__
    WHERE date BETWEEN '__START__' AND '__END__'
),
diffed AS (
    SELECT
        *,
        GREATEST(amount - COALESCE(LAG(amount) OVER w, 0), 0) AS amount_1m,
        GREATEST(volume - COALESCE(LAG(volume) OVER w, 0), 0) AS volume_1m,
        GREATEST(deal_number - COALESCE(LAG(deal_number) OVER w, 0), 0) AS num_trades_1m,
        LAG(close) OVER w AS prev_min_close
    FROM src
    WINDOW w AS (PARTITION BY trading_day, instrument ORDER BY bar_time)
),
signed AS (
    SELECT
        *,
        CASE
            WHEN close > prev_min_close THEN volume_1m
            WHEN close < prev_min_close THEN -volume_1m
            ELSE 0
        END AS obv_step,
        (close / NULLIF(pre_close, 0) - 1) * volume_1m AS pvt_step
    FROM diffed
),
features AS (
    SELECT
        trading_day AS date,
        instrument,
        bar_time,
        minute_no,
        COUNT(*) OVER (PARTITION BY trading_day, instrument) AS minute_count,
        open,
        high,
        low,
        close,
        CAST(volume_1m AS DOUBLE) AS volume,
        CAST(amount_1m AS DOUBLE) AS amount,
        CAST(close / NULLIF(pre_close, 0) - 1 AS DOUBLE) AS return,
        CAST(SUM(amount_1m) OVER day_to_now / NULLIF(SUM(volume_1m) OVER day_to_now, 0) AS DOUBLE) AS vwap,
        CAST(AVG(close) OVER day_to_now AS DOUBLE) AS twap,
        CAST(SUM(obv_step) OVER day_to_now AS DOUBLE) AS obv,
        CAST(SUM(pvt_step) OVER day_to_now AS DOUBLE) AS pvt,
        CAST(num_trades_1m AS DOUBLE) AS num_trades,
        CAST(volume_1m / NULLIF(num_trades_1m, 0) AS DOUBLE) AS vol_per_trade,
        CAST(amount_1m / NULLIF(num_trades_1m, 0) AS DOUBLE) AS amt_per_trade,
        CAST(AVG(close) OVER last5 AS DOUBLE) AS close_ma5,
        CAST(AVG(close) OVER last15 AS DOUBLE) AS close_ma15,
        CAST(AVG(close) OVER last30 AS DOUBLE) AS close_ma30,
        CAST(AVG(close / NULLIF(pre_close, 0) - 1) OVER last5 AS DOUBLE) AS return_ma5,
        CAST(AVG(close / NULLIF(pre_close, 0) - 1) OVER last15 AS DOUBLE) AS return_ma15,
        CAST(AVG(close / NULLIF(pre_close, 0) - 1) OVER last30 AS DOUBLE) AS return_ma30,
        CAST(AVG(volume_1m) OVER last5 AS DOUBLE) AS volume_ma5,
        CAST(AVG(volume_1m) OVER last15 AS DOUBLE) AS volume_ma15,
        CAST(AVG(volume_1m) OVER last30 AS DOUBLE) AS volume_ma30,
        CAST(AVG(volume_1m / NULLIF(num_trades_1m, 0)) OVER last5 AS DOUBLE) AS vol_per_trade_ma5,
        CAST(AVG(volume_1m / NULLIF(num_trades_1m, 0)) OVER last15 AS DOUBLE) AS vol_per_trade_ma15,
        CAST(AVG(volume_1m / NULLIF(num_trades_1m, 0)) OVER last30 AS DOUBLE) AS vol_per_trade_ma30
    FROM signed
    WINDOW
        day_to_now AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
        last5 AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN 4 PRECEDING AND CURRENT ROW),
        last15 AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN 14 PRECEDING AND CURRENT ROW),
        last30 AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN 29 PRECEDING AND CURRENT ROW)
)
,
picked AS (
    SELECT
        *,
        amount AS A_value,
        num_trades AS B_raw,
        volume AS mask_value,
        CASE
            WHEN 60 IS NULL THEN 1
            WHEN 0.9 IS NULL THEN GREATEST(minute_count - 60 + 1, 1)
            ELSE GREATEST(CAST(ROUND(1 + 0.9 * (minute_count - 1) - 60 / 2.0) AS INTEGER), 1)
        END AS start_minute,
        CASE
            WHEN 60 IS NULL THEN minute_count
            WHEN 0.9 IS NULL THEN minute_count
            ELSE LEAST(CAST(ROUND(1 + 0.9 * (minute_count - 1) + 60 / 2.0) AS INTEGER), minute_count)
        END AS end_minute
    FROM features
),
windowed AS (
    SELECT
        *,

        -- B_shift_lag < 0 uses later intraday minutes. This is allowed here because BigAlpha public/private evaluation uses post-close daily factors built from the full trading day.

        LEAD(B_raw, 1) OVER (PARTITION BY date, instrument ORDER BY bar_time) AS B_value
    FROM picked
    WHERE minute_no BETWEEN start_minute AND end_minute
),
ranked AS (
    SELECT
        *,
        (RANK() OVER (PARTITION BY date, instrument ORDER BY mask_value) - 1.0)
            / NULLIF(COUNT(*) OVER (PARTITION BY date, instrument) - 1, 0) AS mask_rank,
        (RANK() OVER (PARTITION BY date, instrument ORDER BY A_value) - 1.0)
            / NULLIF(COUNT(*) OVER (PARTITION BY date, instrument) - 1, 0) AS A_rank,
        MAX(minute_no) OVER (PARTITION BY date, instrument) AS max_minute_no,
        (minute_no - MIN(minute_no) OVER (PARTITION BY date, instrument))
            / NULLIF(MAX(minute_no) OVER (PARTITION BY date, instrument) - MIN(minute_no) OVER (PARTITION BY date, instrument), 0) AS pos01,
        AVG(A_value) OVER (PARTITION BY date, instrument) AS A_mean,
        nanstd(A_value) OVER (PARTITION BY date, instrument) AS A_std,
        AVG(B_value) OVER (PARTITION BY date, instrument) AS B_mean,
        nanstd(B_value) OVER (PARTITION BY date, instrument) AS B_std
    FROM windowed
),
prepared AS (
    SELECT
        *,
        mask_rank >= 0.3 AS keep_mask,
        (A_value - A_mean) / NULLIF(A_std, 0) AS A_z,
        (B_value - B_mean) / NULLIF(B_std, 0) AS B_z,
        MAX(A_value) OVER (PARTITION BY date, instrument) AS A_max,
        MIN(A_value) OVER (PARTITION BY date, instrument) AS A_min
    FROM ranked
),
factor_raw AS (
    SELECT
        1 AS factor_id,
        date,
        instrument,
        SQRT(SUM(CASE WHEN keep_mask THEN POWER(A_z - B_z, 2) END)) AS factor_raw,
        COUNT(CASE WHEN keep_mask THEN 1 END) AS mask_n
    FROM prepared
    GROUP BY date, instrument
)
SELECT factor_id, date, instrument, factor_raw, mask_n
FROM factor_raw
WHERE factor_raw IS NOT NULL

)
SELECT
    date,
    instrument,
    factor_raw * 1 AS factor
FROM raw_factor
WHERE factor_raw IS NOT NULL
''', 'formula': '2|amount|num_trades|60|0.9|volume|high_0.3||Euc_Dist|-1', 'direction': '1', 'source': 'submission/高频因子submission2/高频2_rank12.ipynb'},
    'hf2_rank69': {'sql': r'''
WITH raw_factor AS (

WITH

src AS (
    SELECT
        date::DATE::DATETIME AS trading_day,
        instrument::string AS instrument,
        date AS bar_time,
        open,
        high,
        low,
        close,
        pre_close,
        amount,
        volume,
        deal_number,
        ROW_NUMBER() OVER (PARTITION BY date::DATE::DATETIME, instrument ORDER BY date) AS minute_no
    FROM __BAR1M__
    WHERE date BETWEEN '__START__' AND '__END__'
),
diffed AS (
    SELECT
        *,
        GREATEST(amount - COALESCE(LAG(amount) OVER w, 0), 0) AS amount_1m,
        GREATEST(volume - COALESCE(LAG(volume) OVER w, 0), 0) AS volume_1m,
        GREATEST(deal_number - COALESCE(LAG(deal_number) OVER w, 0), 0) AS num_trades_1m,
        LAG(close) OVER w AS prev_min_close
    FROM src
    WINDOW w AS (PARTITION BY trading_day, instrument ORDER BY bar_time)
),
signed AS (
    SELECT
        *,
        CASE
            WHEN close > prev_min_close THEN volume_1m
            WHEN close < prev_min_close THEN -volume_1m
            ELSE 0
        END AS obv_step,
        (close / NULLIF(pre_close, 0) - 1) * volume_1m AS pvt_step
    FROM diffed
),
features AS (
    SELECT
        trading_day AS date,
        instrument,
        bar_time,
        minute_no,
        COUNT(*) OVER (PARTITION BY trading_day, instrument) AS minute_count,
        open,
        high,
        low,
        close,
        CAST(volume_1m AS DOUBLE) AS volume,
        CAST(amount_1m AS DOUBLE) AS amount,
        CAST(close / NULLIF(pre_close, 0) - 1 AS DOUBLE) AS return,
        CAST(SUM(amount_1m) OVER day_to_now / NULLIF(SUM(volume_1m) OVER day_to_now, 0) AS DOUBLE) AS vwap,
        CAST(AVG(close) OVER day_to_now AS DOUBLE) AS twap,
        CAST(SUM(obv_step) OVER day_to_now AS DOUBLE) AS obv,
        CAST(SUM(pvt_step) OVER day_to_now AS DOUBLE) AS pvt,
        CAST(num_trades_1m AS DOUBLE) AS num_trades,
        CAST(volume_1m / NULLIF(num_trades_1m, 0) AS DOUBLE) AS vol_per_trade,
        CAST(amount_1m / NULLIF(num_trades_1m, 0) AS DOUBLE) AS amt_per_trade,
        CAST(AVG(close) OVER last5 AS DOUBLE) AS close_ma5,
        CAST(AVG(close) OVER last15 AS DOUBLE) AS close_ma15,
        CAST(AVG(close) OVER last30 AS DOUBLE) AS close_ma30,
        CAST(AVG(close / NULLIF(pre_close, 0) - 1) OVER last5 AS DOUBLE) AS return_ma5,
        CAST(AVG(close / NULLIF(pre_close, 0) - 1) OVER last15 AS DOUBLE) AS return_ma15,
        CAST(AVG(close / NULLIF(pre_close, 0) - 1) OVER last30 AS DOUBLE) AS return_ma30,
        CAST(AVG(volume_1m) OVER last5 AS DOUBLE) AS volume_ma5,
        CAST(AVG(volume_1m) OVER last15 AS DOUBLE) AS volume_ma15,
        CAST(AVG(volume_1m) OVER last30 AS DOUBLE) AS volume_ma30,
        CAST(AVG(volume_1m / NULLIF(num_trades_1m, 0)) OVER last5 AS DOUBLE) AS vol_per_trade_ma5,
        CAST(AVG(volume_1m / NULLIF(num_trades_1m, 0)) OVER last15 AS DOUBLE) AS vol_per_trade_ma15,
        CAST(AVG(volume_1m / NULLIF(num_trades_1m, 0)) OVER last30 AS DOUBLE) AS vol_per_trade_ma30
    FROM signed
    WINDOW
        day_to_now AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
        last5 AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN 4 PRECEDING AND CURRENT ROW),
        last15 AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN 14 PRECEDING AND CURRENT ROW),
        last30 AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN 29 PRECEDING AND CURRENT ROW)
)
,
picked AS (
    SELECT
        *,
        volume_ma15 AS A_value,
        volume_ma15 AS B_raw,
        volume_ma30 AS mask_value,
        CASE
            WHEN 238 IS NULL THEN 1
            WHEN 1.0 IS NULL THEN GREATEST(minute_count - 238 + 1, 1)
            ELSE GREATEST(CAST(ROUND(1 + 1.0 * (minute_count - 1) - 238 / 2.0) AS INTEGER), 1)
        END AS start_minute,
        CASE
            WHEN 238 IS NULL THEN minute_count
            WHEN 1.0 IS NULL THEN minute_count
            ELSE LEAST(CAST(ROUND(1 + 1.0 * (minute_count - 1) + 238 / 2.0) AS INTEGER), minute_count)
        END AS end_minute
    FROM features
),
windowed AS (
    SELECT
        *,

        B_raw AS B_value
    FROM picked
    WHERE minute_no BETWEEN start_minute AND end_minute
),
ranked AS (
    SELECT
        *,
        (RANK() OVER (PARTITION BY date, instrument ORDER BY mask_value) - 1.0)
            / NULLIF(COUNT(*) OVER (PARTITION BY date, instrument) - 1, 0) AS mask_rank,
        (RANK() OVER (PARTITION BY date, instrument ORDER BY A_value) - 1.0)
            / NULLIF(COUNT(*) OVER (PARTITION BY date, instrument) - 1, 0) AS A_rank,
        MAX(minute_no) OVER (PARTITION BY date, instrument) AS max_minute_no,
        (minute_no - MIN(minute_no) OVER (PARTITION BY date, instrument))
            / NULLIF(MAX(minute_no) OVER (PARTITION BY date, instrument) - MIN(minute_no) OVER (PARTITION BY date, instrument), 0) AS pos01,
        AVG(A_value) OVER (PARTITION BY date, instrument) AS A_mean,
        nanstd(A_value) OVER (PARTITION BY date, instrument) AS A_std,
        AVG(B_value) OVER (PARTITION BY date, instrument) AS B_mean,
        nanstd(B_value) OVER (PARTITION BY date, instrument) AS B_std
    FROM windowed
),
prepared AS (
    SELECT
        *,
        mask_rank >= 0.5 AS keep_mask,
        (A_value - A_mean) / NULLIF(A_std, 0) AS A_z,
        (B_value - B_mean) / NULLIF(B_std, 0) AS B_z,
        MAX(A_value) OVER (PARTITION BY date, instrument) AS A_max,
        MIN(A_value) OVER (PARTITION BY date, instrument) AS A_min
    FROM ranked
),
factor_raw AS (
    SELECT
        1 AS factor_id,
        date,
        instrument,
        SUM(CASE WHEN keep_mask THEN A_value END) / NULLIF(SUM(A_value), 0) AS factor_raw,
        COUNT(CASE WHEN keep_mask THEN 1 END) AS mask_n
    FROM prepared
    GROUP BY date, instrument
)
SELECT factor_id, date, instrument, factor_raw, mask_n
FROM factor_raw
WHERE factor_raw IS NOT NULL

)
SELECT
    date,
    instrument,
    factor_raw * -1 AS factor
FROM raw_factor
WHERE factor_raw IS NOT NULL
''', 'formula': '1|volume_ma15||238|1.0|volume_ma30|high_0.5|Ratio||0', 'direction': '-1', 'source': 'submission/高频因子submission2/高频2_rank69.ipynb'},
    'hf2_rank77': {'sql': r'''
WITH raw_factor AS (

WITH

src AS (
    SELECT
        date::DATE::DATETIME AS trading_day,
        instrument::string AS instrument,
        date AS bar_time,
        open,
        high,
        low,
        close,
        pre_close,
        amount,
        volume,
        deal_number,
        ROW_NUMBER() OVER (PARTITION BY date::DATE::DATETIME, instrument ORDER BY date) AS minute_no
    FROM __BAR1M__
    WHERE date BETWEEN '__START__' AND '__END__'
),
diffed AS (
    SELECT
        *,
        GREATEST(amount - COALESCE(LAG(amount) OVER w, 0), 0) AS amount_1m,
        GREATEST(volume - COALESCE(LAG(volume) OVER w, 0), 0) AS volume_1m,
        GREATEST(deal_number - COALESCE(LAG(deal_number) OVER w, 0), 0) AS num_trades_1m,
        LAG(close) OVER w AS prev_min_close
    FROM src
    WINDOW w AS (PARTITION BY trading_day, instrument ORDER BY bar_time)
),
signed AS (
    SELECT
        *,
        CASE
            WHEN close > prev_min_close THEN volume_1m
            WHEN close < prev_min_close THEN -volume_1m
            ELSE 0
        END AS obv_step,
        (close / NULLIF(pre_close, 0) - 1) * volume_1m AS pvt_step
    FROM diffed
),
features AS (
    SELECT
        trading_day AS date,
        instrument,
        bar_time,
        minute_no,
        COUNT(*) OVER (PARTITION BY trading_day, instrument) AS minute_count,
        open,
        high,
        low,
        close,
        CAST(volume_1m AS DOUBLE) AS volume,
        CAST(amount_1m AS DOUBLE) AS amount,
        CAST(close / NULLIF(pre_close, 0) - 1 AS DOUBLE) AS return,
        CAST(SUM(amount_1m) OVER day_to_now / NULLIF(SUM(volume_1m) OVER day_to_now, 0) AS DOUBLE) AS vwap,
        CAST(AVG(close) OVER day_to_now AS DOUBLE) AS twap,
        CAST(SUM(obv_step) OVER day_to_now AS DOUBLE) AS obv,
        CAST(SUM(pvt_step) OVER day_to_now AS DOUBLE) AS pvt,
        CAST(num_trades_1m AS DOUBLE) AS num_trades,
        CAST(volume_1m / NULLIF(num_trades_1m, 0) AS DOUBLE) AS vol_per_trade,
        CAST(amount_1m / NULLIF(num_trades_1m, 0) AS DOUBLE) AS amt_per_trade,
        CAST(AVG(close) OVER last5 AS DOUBLE) AS close_ma5,
        CAST(AVG(close) OVER last15 AS DOUBLE) AS close_ma15,
        CAST(AVG(close) OVER last30 AS DOUBLE) AS close_ma30,
        CAST(AVG(close / NULLIF(pre_close, 0) - 1) OVER last5 AS DOUBLE) AS return_ma5,
        CAST(AVG(close / NULLIF(pre_close, 0) - 1) OVER last15 AS DOUBLE) AS return_ma15,
        CAST(AVG(close / NULLIF(pre_close, 0) - 1) OVER last30 AS DOUBLE) AS return_ma30,
        CAST(AVG(volume_1m) OVER last5 AS DOUBLE) AS volume_ma5,
        CAST(AVG(volume_1m) OVER last15 AS DOUBLE) AS volume_ma15,
        CAST(AVG(volume_1m) OVER last30 AS DOUBLE) AS volume_ma30,
        CAST(AVG(volume_1m / NULLIF(num_trades_1m, 0)) OVER last5 AS DOUBLE) AS vol_per_trade_ma5,
        CAST(AVG(volume_1m / NULLIF(num_trades_1m, 0)) OVER last15 AS DOUBLE) AS vol_per_trade_ma15,
        CAST(AVG(volume_1m / NULLIF(num_trades_1m, 0)) OVER last30 AS DOUBLE) AS vol_per_trade_ma30
    FROM signed
    WINDOW
        day_to_now AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
        last5 AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN 4 PRECEDING AND CURRENT ROW),
        last15 AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN 14 PRECEDING AND CURRENT ROW),
        last30 AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN 29 PRECEDING AND CURRENT ROW)
)
,
picked AS (
    SELECT
        *,
        amt_per_trade AS A_value,
        amount AS B_raw,
        vol_per_trade AS mask_value,
        CASE
            WHEN 60 IS NULL THEN 1
            WHEN 0.5 IS NULL THEN GREATEST(minute_count - 60 + 1, 1)
            ELSE GREATEST(CAST(ROUND(1 + 0.5 * (minute_count - 1) - 60 / 2.0) AS INTEGER), 1)
        END AS start_minute,
        CASE
            WHEN 60 IS NULL THEN minute_count
            WHEN 0.5 IS NULL THEN minute_count
            ELSE LEAST(CAST(ROUND(1 + 0.5 * (minute_count - 1) + 60 / 2.0) AS INTEGER), minute_count)
        END AS end_minute
    FROM features
),
windowed AS (
    SELECT
        *,

        LAG(B_raw, 1) OVER (PARTITION BY date, instrument ORDER BY bar_time) AS B_value
    FROM picked
    WHERE minute_no BETWEEN start_minute AND end_minute
),
ranked AS (
    SELECT
        *,
        (RANK() OVER (PARTITION BY date, instrument ORDER BY mask_value) - 1.0)
            / NULLIF(COUNT(*) OVER (PARTITION BY date, instrument) - 1, 0) AS mask_rank,
        (RANK() OVER (PARTITION BY date, instrument ORDER BY A_value) - 1.0)
            / NULLIF(COUNT(*) OVER (PARTITION BY date, instrument) - 1, 0) AS A_rank,
        MAX(minute_no) OVER (PARTITION BY date, instrument) AS max_minute_no,
        (minute_no - MIN(minute_no) OVER (PARTITION BY date, instrument))
            / NULLIF(MAX(minute_no) OVER (PARTITION BY date, instrument) - MIN(minute_no) OVER (PARTITION BY date, instrument), 0) AS pos01,
        AVG(A_value) OVER (PARTITION BY date, instrument) AS A_mean,
        nanstd(A_value) OVER (PARTITION BY date, instrument) AS A_std,
        AVG(B_value) OVER (PARTITION BY date, instrument) AS B_mean,
        nanstd(B_value) OVER (PARTITION BY date, instrument) AS B_std
    FROM windowed
),
prepared AS (
    SELECT
        *,
        mask_rank <= 0.5 AS keep_mask,
        (A_value - A_mean) / NULLIF(A_std, 0) AS A_z,
        (B_value - B_mean) / NULLIF(B_std, 0) AS B_z,
        MAX(A_value) OVER (PARTITION BY date, instrument) AS A_max,
        MIN(A_value) OVER (PARTITION BY date, instrument) AS A_min
    FROM ranked
),
factor_raw AS (
    SELECT
        1 AS factor_id,
        date,
        instrument,
        regr_intercept(CASE WHEN keep_mask THEN A_value END, CASE WHEN keep_mask THEN B_value END) AS factor_raw,
        COUNT(CASE WHEN keep_mask THEN 1 END) AS mask_n
    FROM prepared
    GROUP BY date, instrument
)
SELECT factor_id, date, instrument, factor_raw, mask_n
FROM factor_raw
WHERE factor_raw IS NOT NULL

)
SELECT
    date,
    instrument,
    factor_raw * -1 AS factor
FROM raw_factor
WHERE factor_raw IS NOT NULL
''', 'formula': '2|amt_per_trade|amount|60|0.5|vol_per_trade|low_0.5||Intercept|1', 'direction': '-1', 'source': 'submission/高频因子submission2/高频2_rank77.ipynb'},
    'hf3_rank35': {'sql': r'''
WITH raw_factor AS (

WITH

src AS (
    SELECT
        date::DATE::DATETIME AS trading_day,
        instrument::string AS instrument,
        date AS bar_time,
        open,
        high,
        low,
        close,
        pre_close,
        amount,
        volume,
        deal_number,
        ROW_NUMBER() OVER (PARTITION BY date::DATE::DATETIME, instrument ORDER BY date) AS minute_no
    FROM __BAR1M__
    WHERE date BETWEEN '__START__' AND '__END__'
),
diffed AS (
    SELECT
        *,
        GREATEST(amount - COALESCE(LAG(amount) OVER w, 0), 0) AS amount_1m,
        GREATEST(volume - COALESCE(LAG(volume) OVER w, 0), 0) AS volume_1m,
        GREATEST(deal_number - COALESCE(LAG(deal_number) OVER w, 0), 0) AS num_trades_1m,
        LAG(close) OVER w AS prev_min_close
    FROM src
    WINDOW w AS (PARTITION BY trading_day, instrument ORDER BY bar_time)
),
signed AS (
    SELECT
        *,
        CASE
            WHEN close > prev_min_close THEN volume_1m
            WHEN close < prev_min_close THEN -volume_1m
            ELSE 0
        END AS obv_step,
        (close / NULLIF(pre_close, 0) - 1) * volume_1m AS pvt_step
    FROM diffed
),
features AS (
    SELECT
        trading_day AS date,
        instrument,
        bar_time,
        minute_no,
        COUNT(*) OVER (PARTITION BY trading_day, instrument) AS minute_count,
        open,
        high,
        low,
        close,
        CAST(volume_1m AS DOUBLE) AS volume,
        CAST(amount_1m AS DOUBLE) AS amount,
        CAST(close / NULLIF(pre_close, 0) - 1 AS DOUBLE) AS return,
        CAST(SUM(amount_1m) OVER day_to_now / NULLIF(SUM(volume_1m) OVER day_to_now, 0) AS DOUBLE) AS vwap,
        CAST(AVG(close) OVER day_to_now AS DOUBLE) AS twap,
        CAST(SUM(obv_step) OVER day_to_now AS DOUBLE) AS obv,
        CAST(SUM(pvt_step) OVER day_to_now AS DOUBLE) AS pvt,
        CAST(num_trades_1m AS DOUBLE) AS num_trades,
        CAST(volume_1m / NULLIF(num_trades_1m, 0) AS DOUBLE) AS vol_per_trade,
        CAST(amount_1m / NULLIF(num_trades_1m, 0) AS DOUBLE) AS amt_per_trade,
        CAST(AVG(close) OVER last5 AS DOUBLE) AS close_ma5,
        CAST(AVG(close) OVER last15 AS DOUBLE) AS close_ma15,
        CAST(AVG(close) OVER last30 AS DOUBLE) AS close_ma30,
        CAST(AVG(close / NULLIF(pre_close, 0) - 1) OVER last5 AS DOUBLE) AS return_ma5,
        CAST(AVG(close / NULLIF(pre_close, 0) - 1) OVER last15 AS DOUBLE) AS return_ma15,
        CAST(AVG(close / NULLIF(pre_close, 0) - 1) OVER last30 AS DOUBLE) AS return_ma30,
        CAST(AVG(volume_1m) OVER last5 AS DOUBLE) AS volume_ma5,
        CAST(AVG(volume_1m) OVER last15 AS DOUBLE) AS volume_ma15,
        CAST(AVG(volume_1m) OVER last30 AS DOUBLE) AS volume_ma30,
        CAST(AVG(volume_1m / NULLIF(num_trades_1m, 0)) OVER last5 AS DOUBLE) AS vol_per_trade_ma5,
        CAST(AVG(volume_1m / NULLIF(num_trades_1m, 0)) OVER last15 AS DOUBLE) AS vol_per_trade_ma15,
        CAST(AVG(volume_1m / NULLIF(num_trades_1m, 0)) OVER last30 AS DOUBLE) AS vol_per_trade_ma30
    FROM signed
    WINDOW
        day_to_now AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
        last5 AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN 4 PRECEDING AND CURRENT ROW),
        last15 AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN 14 PRECEDING AND CURRENT ROW),
        last30 AS (PARTITION BY trading_day, instrument ORDER BY bar_time ROWS BETWEEN 29 PRECEDING AND CURRENT ROW)
)
,
picked AS (
    SELECT
        *,
        volume_ma5 AS A_value,
        volume_ma5 AS B_raw,
        amount AS mask_value,
        CASE
            WHEN 120 IS NULL THEN 1
            WHEN 0.9 IS NULL THEN GREATEST(minute_count - 120 + 1, 1)
            ELSE GREATEST(CAST(ROUND(1 + 0.9 * (minute_count - 1) - 120 / 2.0) AS INTEGER), 1)
        END AS start_minute,
        CASE
            WHEN 120 IS NULL THEN minute_count
            WHEN 0.9 IS NULL THEN minute_count
            ELSE LEAST(CAST(ROUND(1 + 0.9 * (minute_count - 1) + 120 / 2.0) AS INTEGER), minute_count)
        END AS end_minute
    FROM features
),
windowed AS (
    SELECT
        *,

        B_raw AS B_value
    FROM picked
    WHERE minute_no BETWEEN start_minute AND end_minute
),
ranked AS (
    SELECT
        *,
        (RANK() OVER (PARTITION BY date, instrument ORDER BY mask_value) - 1.0)
            / NULLIF(COUNT(*) OVER (PARTITION BY date, instrument) - 1, 0) AS mask_rank,
        (RANK() OVER (PARTITION BY date, instrument ORDER BY A_value) - 1.0)
            / NULLIF(COUNT(*) OVER (PARTITION BY date, instrument) - 1, 0) AS A_rank,
        MAX(minute_no) OVER (PARTITION BY date, instrument) AS max_minute_no,
        (minute_no - MIN(minute_no) OVER (PARTITION BY date, instrument))
            / NULLIF(MAX(minute_no) OVER (PARTITION BY date, instrument) - MIN(minute_no) OVER (PARTITION BY date, instrument), 0) AS pos01,
        AVG(A_value) OVER (PARTITION BY date, instrument) AS A_mean,
        nanstd(A_value) OVER (PARTITION BY date, instrument) AS A_std,
        AVG(B_value) OVER (PARTITION BY date, instrument) AS B_mean,
        nanstd(B_value) OVER (PARTITION BY date, instrument) AS B_std
    FROM windowed
),
prepared AS (
    SELECT
        *,
        mask_rank >= 0.3 AS keep_mask,
        (A_value - A_mean) / NULLIF(A_std, 0) AS A_z,
        (B_value - B_mean) / NULLIF(B_std, 0) AS B_z,
        MAX(A_value) OVER (PARTITION BY date, instrument) AS A_max,
        MIN(A_value) OVER (PARTITION BY date, instrument) AS A_min
    FROM ranked
),
factor_raw AS (
    SELECT
        1 AS factor_id,
        date,
        instrument,
        skewness(CASE WHEN keep_mask THEN A_value END) AS factor_raw,
        COUNT(CASE WHEN keep_mask THEN 1 END) AS mask_n
    FROM prepared
    GROUP BY date, instrument
)
SELECT factor_id, date, instrument, factor_raw, mask_n
FROM factor_raw
WHERE factor_raw IS NOT NULL

)
SELECT
    date,
    instrument,
    factor_raw * -1 AS factor
FROM raw_factor
WHERE factor_raw IS NOT NULL
''', 'formula': '1|volume_ma5||120|0.9|amount|high_0.3|Skew||0', 'direction': '-1', 'source': 'submission/高频因子submission3/高频3_rank35.ipynb'},
}
ALL_LABELS = ['hf2_rank4', 'hf2_rank12', 'hf2_rank69', 'hf2_rank77', 'hf3_rank35']
FACTOR_WEIGHTS = {'hf2_rank4': 1.0, 'hf2_rank12': 1.0, 'hf2_rank69': 1.0, 'hf2_rank77': 1.0, 'hf3_rank35': 1.0}
CHUNK_DAYS = 92


def _date_chunks(start_date, end_date):
    """Yield non-overlapping timestamp chunks without splitting a trading day."""
    cursor = pd.Timestamp(start_date)
    final_end = pd.Timestamp(end_date)
    while cursor <= final_end:
        chunk_end = min(
            cursor.normalize() + pd.Timedelta(days=CHUNK_DAYS) - pd.Timedelta(seconds=1),
            final_end,
        )
        yield (
            cursor.strftime("%Y-%m-%d %H:%M:%S"),
            chunk_end.strftime("%Y-%m-%d %H:%M:%S"),
        )
        cursor = chunk_end + pd.Timedelta(seconds=1)


def _single_factor(dai, spec, bar1m, chunk_start, chunk_end):
    """Run one baked factor SQL for one date chunk."""
    sql = (spec["sql"]
           .replace("__BAR1M__", bar1m)
           .replace("__START__", chunk_start)
           .replace("__END__", chunk_end))
    df = dai.query(
        sql,
        filters={"date": [chunk_start, chunk_end]},
        compression=True,
    ).df()
    if len(df) == 0:
        return pd.DataFrame(columns=["date", "instrument", "value"])

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["value"] = pd.to_numeric(df["factor"], errors="coerce")
    df["value"] = df["value"].replace([np.inf, -np.inf], np.nan)
    df = df[["date", "instrument", "value"]].dropna(subset=["value"])
    return df.groupby(["date", "instrument"], as_index=False, sort=False)["value"].mean()


def _stock_pool(dai, chunk_start, chunk_end):
    pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [chunk_start, chunk_end]},
        compression=True,
    ).df()
    if len(pool) == 0:
        return pd.DataFrame(columns=["date", "instrument"])
    pool["date"] = pd.to_datetime(pool["date"])
    pool["instrument"] = pool["instrument"].astype(str)
    return pool.drop_duplicates(["date", "instrument"])


def main(datasources, start_date, end_date):
    import dai

    bar1m = datasources.get("bar1m", "bigalpha_2026_stock_bar1m")
    chunks = list(_date_chunks(start_date, end_date))
    chunk_results = []

    for chunk_i, (chunk_start, chunk_end) in enumerate(chunks, 1):
        chunk_t0 = time.time()
        print(
            f"[combo] chunk {chunk_i}/{len(chunks)}: {chunk_start} -> {chunk_end}",
            flush=True,
        )
        pool = _stock_pool(dai, chunk_start, chunk_end)
        if len(pool) == 0:
            print("[combo] stock pool is empty; chunk skipped", flush=True)
            continue

        rank_parts = []
        for factor_i, label in enumerate(ALL_LABELS, 1):
            factor_t0 = time.time()
            print(
                f"[combo]   factor {factor_i}/{len(ALL_LABELS)}: {label}",
                flush=True,
            )
            factor = _single_factor(
                dai,
                FACTOR_SPECS[label],
                bar1m,
                chunk_start,
                chunk_end,
            )
            if len(factor) == 0:
                print(f"[combo]   {label} empty; skipped", flush=True)
                del factor
                gc.collect()
                continue

            factor = factor.merge(pool, how="inner", on=["date", "instrument"])
            if len(factor) == 0:
                print(f"[combo]   {label} empty after stock-pool filter; skipped", flush=True)
                del factor
                gc.collect()
                continue

            factor["score"] = factor.groupby("date")["value"].rank(pct=True)
            rank_parts.append(
                factor[["date", "instrument", "score"]]
                .dropna(subset=["score"])
                .assign(weight=float(FACTOR_WEIGHTS[label]))
            )
            print(
                f"[combo]   {label} rows={len(factor)} "
                f"elapsed={time.time() - factor_t0:.1f}s",
                flush=True,
            )
            del factor
            gc.collect()

        if rank_parts:
            long_chunk = pd.concat(rank_parts, ignore_index=True, copy=False)
            long_chunk["weighted_score"] = long_chunk["score"] * long_chunk["weight"]
            combined_chunk = long_chunk.groupby(
                ["date", "instrument"], as_index=False, sort=False
            ).agg(
                weighted_score=("weighted_score", "sum"),
                available_weight=("weight", "sum"),
            )
            combined_chunk["factor"] = (
                combined_chunk["weighted_score"] / combined_chunk["available_weight"]
            )
            combined_chunk = combined_chunk[["date", "instrument", "factor"]]
            chunk_results.append(combined_chunk)
            print(
                f"[combo] chunk rows={len(combined_chunk)} "
                f"elapsed={time.time() - chunk_t0:.1f}s",
                flush=True,
            )
            del long_chunk, combined_chunk
        else:
            print("[combo] no valid factor in this chunk", flush=True)

        del rank_parts, pool
        gc.collect()

    if not chunk_results:
        return pd.DataFrame(columns=["date", "instrument", "factor"])

    result = pd.concat(chunk_results, ignore_index=True, copy=False)
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce")
    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan)
    result = result.dropna(subset=["factor"])
    result = result.drop_duplicates(["date", "instrument"], keep="last")
    return result.reset_index(drop=True)[["date", "instrument", "factor"]]


if __name__ == "__main__":
    from bigmodule import M
    import dai

    datasources = {"bar1m": "bigalpha_2026_stock_bar1m"}
    start_date, end_date = "2024-01-01 00:00:00", "2024-12-31 23:59:59"
    factor_data = main(datasources, start_date, end_date)
    print("factor rows:", len(factor_data))
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()
    M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )
